In [15]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Configure S3 client to use unsigned requests
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = 'broad-references'
prefix = 'hg38'

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        print(obj['Key'])



hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf
hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf.idx
hg38/v0/1000G_omni2.5.hg38.vcf.gz
hg38/v0/1000G_omni2.5.hg38.vcf.gz.tbi
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz.tbi
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf.idx
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz.tbi
hg38/v0/CrossSpeciesContamination/ContaminantNormalizationFactors.txt
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.dict
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.fai
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.img
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.min2k.db
hg38/v0/CrossSpec

In [21]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import gzip
import io

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

bucket = "1000genomes"
key = "1000G_2504_high_coverage/working/20201028_3202_raw_GT_with_annot/20201028_CCDG_14151_B01_GRM_WGS_2020-08-05_chr21.recalibrated_variants.vcf.gz"

obj = s3.get_object(Bucket=bucket, Key=key, Range="bytes=0-300000")
data = gzip.GzipFile(fileobj=io.BytesIO(obj["Body"].read()))

count = 0
for line in data:
    print(line.decode("utf-8").strip())
    count += 1
    if count >= 50:
        break

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed (VQSR)">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele at this location">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=VQSRTrancheINDEL99.00to100.00+,Description="Truth sensitivity tranche level for INDEL model at VQS Lod < -85077.1808">
##FILTER=<ID=VQSRTrancheINDEL99.00to100.00,Description="Truth sensitivity tranche level for INDEL model at VQS Lod: -85077.1808 <= x < -2.3474">
##FILTER=<ID=VQSRTrancheSNP99.80to100.00+,Description="Truth sensitivity tranche level for SNP model at VQS Lod < -343144.6144">
##FILTER=<ID=VQSRTrancheSNP99.80to100.00,Description="Truth sensitivity tranche level for SNP model at VQS Lod: -343144.6144 <= x < -13.4687">
##FORMAT=<ID=AB,Number=1,Type=Float,Description="Allele balance for each het genotype">
##FORMAT=<ID=AD,Number=.,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,T

In [5]:
import s3fs
import json

# Create an S3 filesystem object with unsigned access
fs = s3fs.S3FileSystem(anon=True)

s3_file = "s3://1000genomes/1000G_2504_high_coverage/working/20201028_3202_phased/phased-manifest_July2021.tsv"

with fs.open(s3_file, 'r') as f:
    metadata = json.load(f)

# Inspect top-level keys
print(metadata.keys())


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Create unsigned S3 client
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "1000G_2504_high_coverage/working"

# Get list of VCFs and TBIs
print(f"Listing files under s3://{bucket}/{prefix} ...\n")
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        size = obj["Size"]
        if key.endswith(".vcf.gz") or key.endswith(".vcf.gz.tbi"):
            vcf_info.append((key, size))

# Print summary
for key, size in sorted(vcf_info):
    flag = "✅ OK" if size > 0 else "❌ EMPTY"
    print(f"{key:<100} {size/1e6:8.2f} MB   {flag}")

# Optional: Check if every VCF has a corresponding .tbi file
print("\n=== Index pairing check ===")
vcf_bases = {k.replace(".tbi", "") for k, _ in vcf_info}
for base in vcf_bases:
    has_vcf = any(k == base for k, _ in vcf_info)
    has_tbi = any(k == base + ".tbi" for k, _ in vcf_info)
    if has_vcf and not has_tbi:
        print(f"⚠️ Missing index for: {base}")


Listing files under s3://1000genomes/1000G_2504_high_coverage/working ...

1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.0

In [4]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Unsigned S3 client (for public buckets like 1000 Genomes)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "release/20130502/ALL.chr21"

# Use paginator to list all files
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith(".vcf.gz.tbi"):
            # Check if the index exists by looking for a .tbi file in the same prefix
            index_key = key + ".tbi"
            try:
                s3.head_object(Bucket=bucket, Key=index_key)
                has_index = True
            except s3.exceptions.ClientError:
                has_index = False

            vcf_info.append({
                "vcf": key,
                "size_MB": obj['Size'] / 1024**2,
                "has_index": has_index
            })

# Print results
for info in vcf_info:
    print(f"{info['vcf']}  {info['size_MB']:.2f} MB  Index: {info['has_index']}")


release/20130502/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi  0.03 MB  Index: False


In [14]:
import boto3
import gzip
from io import BytesIO
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
key = "1000G_2504_high_coverage/working/20201028_3202_raw_GT_with_annot/20201028_CCDG_14151_B01_GRM_WGS_2020-08-05_chr8.recalibrated_variants.vcf.gz"

# Get the object (you can still limit the first few KB to save bandwidth)
resp = s3.get_object(Bucket=bucket, Key=key, Range='bytes=0-65535')  # 64 KB
compressed_bytes = resp['Body'].read()

# Decompress in memory
with gzip.open(BytesIO(compressed_bytes), 'rt') as f:
    for line in f:
        if line.startswith('#CHROM'):
            print("Found header line:", line.strip())
            break
        elif line.startswith('##'):
            # Optional: print contig lines or other metadata
            print(line.strip())


##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed (VQSR)">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele at this location">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=VQSRTrancheINDEL99.00to100.00+,Description="Truth sensitivity tranche level for INDEL model at VQS Lod < -85077.1808">
##FILTER=<ID=VQSRTrancheINDEL99.00to100.00,Description="Truth sensitivity tranche level for INDEL model at VQS Lod: -85077.1808 <= x < -2.3474">
##FILTER=<ID=VQSRTrancheSNP99.80to100.00+,Description="Truth sensitivity tranche level for SNP model at VQS Lod < -343144.6144">
##FILTER=<ID=VQSRTrancheSNP99.80to100.00,Description="Truth sensitivity tranche level for SNP model at VQS Lod: -343144.6144 <= x < -13.4687">
##FORMAT=<ID=AB,Number=1,Type=Float,Description="Allele balance for each het genotype">
##FORMAT=<ID=AD,Number=.,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,T